In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/otto-recommender-system/sample_submission.csv
/kaggle/input/competitions/otto-recommender-system/test.jsonl
/kaggle/input/competitions/otto-recommender-system/train.jsonl


Cellule 1 : paramètres et chargement

In [3]:
import json, gc, heapq
from collections import defaultdict
import numpy as np, pandas as pd

TRAIN = "/kaggle/input/competitions/otto-recommender-system/train.jsonl"
TEST  = "/kaggle/input/competitions/otto-recommender-system/test.jsonl"

MAX_SESSIONS  = 1_000_000                 # mettez la même valeur que dans votre version 12
CHUNK         = 20_000                  # sessions traitées par bloc (mémoire)
LAST_N        = 20                      # derniers événements de la session utilisés
TOP_PER_ITEM  = 20                      # voisins gardés par article
SEEN_N        = 10                      # articles de la session mis en tête de liste
WINDOW_MS     = 24 * 3600 * 1000        # fenêtre de co-visitation : 1 jour (10**15 pour désactiver)
BUY_WINDOW_MS = 14 * 24 * 3600 * 1000   # fenêtre pour la matrice paniers/commandes
TYPE_W        = {0: 1, 1: 6, 2: 3}      # poids par type (clic, panier, commande) ; tout à 1 pour désactiver
SEEN_TYPE_W   = {0: 1, 1: 6, 2: 3}      # poids par type pour choisir les articles de la session (carts/orders)
BUY_W         = 2.0                     # poids de la matrice paniers/commandes (à tester : 1, 2, 4)
SEPARATE      = True                    # False = même liste pour clicks/carts/orders (comme avant)
MIN_COUNT, W2V_DIM, TOPN_W2V = 5, 64, 20
TYPE_ID = {"clicks": 0, "carts": 1, "orders": 2}

def read_sessions(path, max_sessions=None):
    sess, aids, tss, tys = [], [], [], []
    with open(path) as f:
        for i, line in enumerate(f):
            if max_sessions and i >= max_sessions: break
            s = json.loads(line)
            for e in s["events"]:
                sess.append(s["session"]); aids.append(e["aid"])
                tss.append(e["ts"]); tys.append(TYPE_ID[e["type"]])
    return pd.DataFrame({"session": np.array(sess, dtype=np.int64),
                         "aid": np.array(aids, dtype=np.int64),
                         "ts": np.array(tss, dtype=np.int64),
                         "type": np.array(tys, dtype=np.int8)})

train = read_sessions(TRAIN, MAX_SESSIONS)
popular = train["aid"].value_counts().index[:20].tolist()
print(train.shape)

(39889600, 4)


Cellule 2 : matrices de co-visitation

In [4]:
import numpy as np, pandas as pd, gc, time

def prepare(train, only_buy=False):
    """Une seule fois : trier, puis garder les LAST_N derniers événements par session."""
    d = train[train["type"] > 0] if only_buy else train
    d = d.sort_values(["session", "ts"], kind="stable")
    d = d.groupby("session", sort=False).tail(LAST_N)
    return d.reset_index(drop=True)

def _merge_counts(keys_list, w_list):
    keys = np.concatenate(keys_list); w = np.concatenate(w_list)
    u, inv = np.unique(keys, return_inverse=True)
    return u, np.bincount(inv, weights=w)

def build_covisit_fast(d, window_ms, type_w, M, chunk=CHUNK):
    w_arr = np.array([type_w[0], type_w[1], type_w[2]], dtype=np.float64)
    sess = d["session"].values; aid = d["aid"].values.astype(np.int64)
    ts = d["ts"].values; ty = d["type"].values
    starts = np.flatnonzero(np.r_[True, sess[1:] != sess[:-1]])
    bounds = [starts[k] if k < len(starts) else len(d) for k in range(0, len(starts) + chunk, chunk)]
    bounds = sorted(set(bounds + [len(d)]))
    acc_k, acc_w = [], []
    for lo, hi in zip(bounds[:-1], bounds[1:]):
        s_, a_, t_, y_ = sess[lo:hi], aid[lo:hi], ts[lo:hi], ty[lo:hi]
        ks, ws = [], []
        for sh in range(1, LAST_N):
            if sh >= len(s_): break
            m = (s_[:-sh] == s_[sh:]) & ((t_[sh:] - t_[:-sh]) < window_ms) & (a_[:-sh] != a_[sh:])
            if not m.any(): continue
            ai, aj = a_[:-sh][m], a_[sh:][m]
            ks.append(ai * M + aj); ws.append(w_arr[y_[sh:][m]])
            ks.append(aj * M + ai); ws.append(w_arr[y_[:-sh][m]])
        if ks:
            u, sm = _merge_counts(ks, ws)
            acc_k.append(u); acc_w.append(sm)
        if len(acc_k) >= 6:                                   # fusionner de temps en temps
            u, sm = _merge_counts(acc_k, acc_w); acc_k, acc_w = [u], [sm]
        del ks, ws; gc.collect()
    keys, w = _merge_counts(acc_k, acc_w)
    ax, ay = keys // M, keys % M
    order = np.lexsort((-w, ax)); ax, ay, w = ax[order], ay[order], w[order]
    g_start = np.flatnonzero(np.r_[True, ax[1:] != ax[:-1]])
    g_id = np.repeat(np.arange(len(g_start)), np.diff(np.r_[g_start, len(ax)]))
    rank = np.arange(len(ax)) - g_start[g_id]
    keep = rank < TOP_PER_ITEM
    ax, ay, w = ax[keep], ay[keep], w[keep]
    g_start = np.flatnonzero(np.r_[True, ax[1:] != ax[:-1]])
    ends = np.r_[g_start[1:], len(ax)]
    return {int(ax[s]): list(zip(ay[s:e].tolist(), w[s:e].tolist())) for s, e in zip(g_start, ends)}

t0 = time.time()
M = int(train["aid"].max()) + 1
d_all = prepare(train, only_buy=False)
cv_all = build_covisit_fast(d_all, WINDOW_MS, TYPE_W, M); del d_all
d_buy = prepare(train, only_buy=True)
cv_buy = build_covisit_fast(d_buy, BUY_WINDOW_MS, {0: 1, 1: 1, 2: 1}, M); del d_buy
print(len(cv_all), len(cv_buy), f"{time.time() - t0:.0f} s")

1057940 510834 144 s


Cellule 3 : word2vec

In [6]:
from gensim.models import Word2Vec     # si ImportError : !pip install gensim

sessions = (train.sort_values(["session", "ts"]).groupby("session")["aid"]
            .apply(lambda s: [str(a) for a in s]).tolist())
w2v = Word2Vec(sessions, vector_size=W2V_DIM, window=5, min_count=MIN_COUNT,
               sg=1, workers=4, epochs=3, seed=0)
del sessions; gc.collect()

kv = w2v.wv
vecs = kv.vectors / np.linalg.norm(kv.vectors, axis=1, keepdims=True)
keys = np.array([int(k) for k in kv.index_to_key])
w2v_nb = {}
k = min(TOPN_W2V, len(keys) - 1)
for s in range(0, len(vecs), 1000):
    sims = vecs[s:s + 1000] @ vecs.T
    top = np.argpartition(-sims, k, axis=1)[:, :k + 1]
    for i in range(sims.shape[0]):
        idx = top[i][np.argsort(-sims[i, top[i]])]
        idx = [j for j in idx if j != s + i][:k]
        w2v_nb[int(keys[s + i])] = [(int(keys[j]), float(sims[i, j])) for j in idx]
print(len(w2v_nb), "articles avec voisins word2vec")

813400 articles avec voisins word2vec


Cellule 4 : recommandations et soumission

In [7]:
def build_list(first, scores, seen):
    fs = set(first)
    top = heapq.nlargest(20 + len(fs), scores.items(), key=lambda x: x[1])
    final = list(first) + [b for b, _ in top if b not in fs]
    final = final[:20]
    if len(final) < 20:                                       # compléter avec les voisins word2vec
        extra = defaultdict(float)
        for rank, a in enumerate(seen):
            for b, s in w2v_nb.get(a, []):
                extra[b] += s / (1 + rank)
        have = set(final)
        for b, _ in sorted(extra.items(), key=lambda x: -x[1]):
            if len(final) >= 20: break
            if b not in have: final.append(b); have.add(b)
    have = set(final)
    for p in popular:                                         # dernier recours : populaires
        if len(final) >= 20: break
        if p not in have: final.append(p); have.add(p)
    return [int(x) for x in final[:20]]

def recommend_both(events):
    aids  = [e["aid"] for e in events]
    types = [TYPE_ID[e["type"]] for e in events]
    seen = list(dict.fromkeys(reversed(aids)))[:LAST_N]       # derniers articles uniques

    sc_all, sc_buy = defaultdict(float), defaultdict(float)
    for rank, a in enumerate(seen):
        r = 1.0 / (1 + rank)
        for b, w in cv_all.get(a, []): sc_all[b] += w * r
        for b, w in cv_buy.get(a, []): sc_buy[b] += w * r

    list_clicks = build_list(seen[:SEEN_N], sc_all, seen)
    if not SEPARATE:
        return list_clicks, list_clicks

    own = defaultdict(float)                                  # articles de la session, pondérés par type et récence
    for i, (a, t) in enumerate(zip(reversed(aids), reversed(types))):
        own[a] += SEEN_TYPE_W[t] / (1 + i)
    first_buy = [a for a, _ in sorted(own.items(), key=lambda x: -x[1])][:SEEN_N]

    sc_b = defaultdict(float, sc_all)
    for b, v in sc_buy.items(): sc_b[b] += BUY_W * v
    list_buys = build_list(first_buy, sc_b, seen)
    return list_clicks, list_buys

with open(TEST) as f, open("submission.csv", "w", newline="") as out:
    out.write("session_type,labels\n")
    for line in f:
        s = json.loads(line)
        lc, lb = recommend_both(s["events"])
        sc = " ".join(map(str, lc)); sb = " ".join(map(str, lb))
        sid = s["session"]
        out.write(f"{sid}_clicks,{sc}\n{sid}_carts,{sb}\n{sid}_orders,{sb}\n")
print("submission.csv écrit")

submission.csv écrit
